In [0]:
from json import loads
from io import StringIO
import requests
import logging
from typing import List, Dict, Optional
from datetime import datetime
import pandas as pd
import os

In [0]:
logger = logging.getLogger(__name__)

In [0]:
def _insert_dataframe_to_db(df: pd.DataFrame, db_name: str, schema_name: str, table_name: str):
    try:
        spark.createDataFrame(df).write.mode('overwrite').format('delta').option('overwriteSchema', 'true').saveAsTable(f'{db_name}.{schema_name}.{table_name}')
        logger.info(f'Inserted {df.shape[0]} rows')
    except Exception as e:
        logger.error(f"Error occurred while inserting symbol changes data into table {table_name}: {e}")
        raise Exception(f"Failed to insert symbol changes data into table {table_name}")

In [0]:
def _download_symbol_changes(db_name: str, schema_name: str, table_name: str):
    session = requests.Session()
    headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, \'like Gecko) \'Chrome/80.0.3987.149 Safari/537.36'}
    cookies = {}

    symbol_changes_url = "https://nsearchives.nseindia.com/content/equities/symbolchange.csv"
    try:
        response = session.get(symbol_changes_url, headers = headers, timeout = 10, cookies = cookies)
    except:
        logger.error(f"Error occurred while downloading symbol changes data from {symbol_changes_url}")
        raise Exception('Failed to download symbol changes data')
    
    try:
        changes_df = pd.read_csv(StringIO(response.text), header = None, names = ['company_name', 'old_symbol', 'new_symbol', 'effective_date'])
    except:
        logger.error(f"Error occurred while parsing symbol changes content at {symbol_changes_url}")
        raise Exception(f'Failed to parse changes symbols content at {symbol_changes_url}')

    changes_df = changes_df[changes_df['old_symbol'] != changes_df['new_symbol']]
    symbol_change_map = changes_df[['old_symbol', 'new_symbol']].set_index('old_symbol')['new_symbol'].to_dict()

    def _latest_symbol_mapper(ticker):
        nxt_ticker = symbol_change_map.get(ticker, None)
        while nxt_ticker in symbol_change_map:
            nxt_ticker = symbol_change_map.get(nxt_ticker, None)
        return nxt_ticker if nxt_ticker else ticker

    changes_df['latest_symbol'] = changes_df['old_symbol'].apply(_latest_symbol_mapper)
    changes_df['record_timestamp'] = datetime.now()
    
    _insert_dataframe_to_db(changes_df, db_name, schema_name, table_name)
    
    return

In [0]:
_download_symbol_changes('indian_market', 'nse_india', 'symbol_changes')

In [0]:
def _download_all_tickers(db_name: str, schema_name: str, table_name: str):
    session = requests.Session()
    headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, \'like Gecko) \'Chrome/80.0.3987.149 Safari/537.36'}
    cookies = {}
    
    all_tickers_url = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"
    
    try:
        response = session.get(all_tickers_url, headers = headers, timeout = 10, cookies = cookies)
    except Exception as e:
        logger.error(f"Error occurred while downloading all tickers data from {all_tickers_url}: {e}")
        raise Exception('Failed to download all tickers data')

    try:
        all_df = pd.read_csv(StringIO(response.text))
        all_df.columns = all_df.columns.str.strip().str.lower().str.split().str.join('_')
        all_df = all_df[['symbol', 'date_of_listing', 'isin_number']].rename(columns = {'symbol' : 'ticker', 
                                                                                        'date_of_listing' : 'listing_date', 
                                                                                        'isin_number' : 'ISIN'
                                                                            })
        all_df['record_timestamp'] = datetime.now()
        all_df['listing_date'] = pd.to_datetime(all_df['listing_date'], format='%d-%b-%Y').dt.strftime('%Y-%m-%d')
        all_df['ticker'] = all_df['ticker'].str.strip().str.upper()
    except Exception as e:
        logger.error(f"Error occurred while parsing all tickers content at {all_tickers_url}: {e}")
        raise Exception(f'Failed to parse all tickers content at {all_tickers_url}')

    _insert_dataframe_to_db(all_df, db_name, schema_name, table_name)

In [0]:
_download_all_tickers('indian_market', 'nse_india', 'active_ticker_list')

In [0]:
# def _download_recent_listings(db_name: str, schema_name: str, table_name: str):
#     session = requests.Session()
#     headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, \'like Gecko) \'Chrome/80.0.3987.149 Safari/537.36'}
#     cookies = {}


#     recent_listing_url = "https://www.nseindia.com/api/new-listing-today?index=RecentListing"
#     try:
#         response = session.get(recent_listing_url, headers = headers, cookies = cookies)
#     except Exception as e:
#         logger.error(f"Error occurred while downloading recent listings data from {recent_listing_url}: {e}")
#         raise Exception('Failed to download recent listings data')
#     try:
#         df = pd.DataFrame(response.json().get('data', []))
#     except Exception as e:
#         logger.error(f"Error occurred while parsing recent listings content at {recent_listing_url}: {e}")
#         raise Exception(f'Failed to parse recent listings content at {recent_listing_url}')

#     df = df[df['series'] == 'EQ'][['symbol', 'listing_date', 'isin']].rename(columns = {'symbol' : 'ticker', 'isin' : 'ISIN'})  
#     df['record_timestamp'] = datetime.now()
#     df['listing_date'] = pd.to_datetime(df['listing_date'], format='%d-%b-%Y').dt.strftime('%Y-%m-%d')
#     df['ticker'] = df['ticker'].str.upper().str.strip()

#     try:
#         spark.createDataFrame(df).createOrReplaceTempView('staging_tickers')
#         res = spark.sql(f"""  
#                         MERGE INTO {db_name}.{schema_name}.{table_name} trgt
#                         USING staging_tickers src ON trgt.ticker = src.ticker
#                         WHEN NOT MATCHED THEN INSERT *
#                     """)
#         logger.info(f'Inserted {res.collect()[0][0]} rows')
#     except Exception as e:
#         logger.error(f"Error occurred while inserting recent listings data into table {table_name}: {e}")
#         raise Exception(f"Failed to insert recent listings data into table {table_name}")

In [0]:
# _download_recent_listings('indian_market', 'nse_india', 'active_ticker_list')